# VFM Course — Dataset Download Notebook

Downloads the three datasets used in the Day 1 probing challenge and mini-projects:

| Dataset | Purpose | Approx. size | Access |
|---|---|---|---|
| **PathMNIST** | fast iteration / Python on-ramp | ~200 MB | open, auto-download |
| **CAMELYON17-WILDS** | domain shift / stain-normalization projects | ~11 GB | open, auto-download |
| **PANDA** | ISUP grading, multi-institution generalization | ~350+ GB (raw WSIs) | **Kaggle account + competition rules acceptance required** |

**Run this on the server, not on a laptop.** Set `DATA_ROOT` below to a volume with enough free space
(at least ~400 GB if you're pulling full-resolution PANDA; far less if you only take a curated subset — see
the PANDA section for a lighter option).

Each section is independent — run only the cells for the dataset(s) you need right now.

## 0. Setup

Installs the packages needed for the three downloaders:
- [`medmnist`](https://github.com/MedMNIST/MedMNIST) for PathMNIST
- [`wilds`](https://wilds.stanford.edu/) for the CAMELYON17-WILDS distribution
- [`kaggle`](https://github.com/Kaggle/kaggle-api) for PANDA (Kaggle-hosted competition data)

In [ ]:
# !pip install --upgrade pip

In [ ]:
# !pip install -r requirements.txt --user

In [ ]:
from pathlib import Path
import pandas as pd

# --- EDIT THIS ---------------------------------------------------------
DATA_ROOT = Path("/home/shared/data/")  # <- point this at your server's data volume
# ------------------------------------------------------------------------

PATHMNIST_DIR = DATA_ROOT / "pathmnist"
CAMELYON17_DIR = DATA_ROOT / "camelyon17"
PANDA_DIR = DATA_ROOT / "panda"

for d in [PATHMNIST_DIR, CAMELYON17_DIR, PANDA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Data root:", DATA_ROOT.resolve())
print("Free space (GB):", round(__import__("shutil").disk_usage(DATA_ROOT).free / 1e9, 1))

## 1. PathMNIST

Part of the [MedMNIST v2](https://medmnist.com/) collection: 9-class colon pathology tissue
classification, derived from the NCT-CRC-HE-100K dataset. Ships in 28×28 and 64×64/128×128/224×224
resolutions — grab 224×224 if you want something closer to real patch inputs for a VFM encoder,
or 28×28 for pure speed during a live session.

Fully automatic download, no account needed, ~200 MB total for all splits at 224px.

In [ ]:
from medmnist import PathMNIST
from medmnist import INFO

# size options: 28, 64, 128, 224 (pixels). 224 is closest to typical VFM patch inputs.
IMAGE_SIZE = 224

for split in ["train", "val", "test"]:
    ds = PathMNIST(split=split, download=True, root=str(PATHMNIST_DIR), size=IMAGE_SIZE)
    print(f"{split:5s}: {len(ds):>7,} images  |  classes: {len(INFO['pathmnist']['label'])}")

print("\nSaved to:", PATHMNIST_DIR.resolve())

In [ ]:
import numpy as np

# save a subset
pathmnist_base_dir = "/home/shared/data/pathmnist/"
pathmnist_path = pathmnist_base_dir + "pathmnist_224.npz"
data = np.load(pathmnist_path)

LABEL_NAMES = {
    0: "adipose",
    1: "background",
    2: "debris",
    3: "lymphocytes",
    4: "mucus",
    5: "smooth muscle",
    6: "normal colon mucosa",
    7: "cancer-associated stroma",
    8: "colorectal adenocarcinoma epithelium",
}


N_TOTAL = 1000
rng = np.random.default_rng(42)

splits = ["train", "val", "test"]
split_sizes = {split: len(data[f"{split}_labels"]) for split in splits}
total_size = sum(split_sizes.values())
classes = sorted(LABEL_NAMES)

subset = {}
for split in splits:
    labels = data[f"{split}_labels"].reshape(-1)
    images = data[f"{split}_images"]

    n_split = round(N_TOTAL * split_sizes[split] / total_size)
    n_per_class = n_split // len(classes)
    remainder = n_split - n_per_class * len(classes)

    split_idxs = []
    for i, label in enumerate(classes):
        class_idxs = np.where(labels == label)[0]
        n = n_per_class + (1 if i < remainder else 0)
        split_idxs.append(rng.choice(class_idxs, size=min(n, len(class_idxs)), replace=False))
    split_idxs = np.concatenate(split_idxs)
    rng.shuffle(split_idxs)

    subset[f"{split}_images"] = images[split_idxs]
    subset[f"{split}_labels"] = labels[split_idxs]

subset_path = pathmnist_base_dir + "pathmnist_224_subset1000.npz"
np.savez(subset_path, **subset)

print("Saved subset to:", subset_path)
for split in splits:
    print(f"{split}: {len(subset[f'{split}_labels'])} samples")

## 2. CAMELYON17-WILDS

The WILDS-curated distribution of CAMELYON17: pre-extracted 96×96 patches with hospital-of-origin
labels attached, purpose-built for domain-shift benchmarking (in-distribution vs. out-of-distribution
by medical center). This is what you want for the *"does the model generalize across institutions?"*
and *"does stain normalization change the representation?"* mini-projects — no WSI tiling required.

~10.7 GB, ~456k patches, 5 hospitals. CC0 license, open download (no account needed).

In [ ]:
!pip install wilds --user

In [ ]:
from wilds import get_dataset

dataset = get_dataset(dataset="camelyon17", download=True, root_dir=str(CAMELYON17_DIR))

print("Dataset:", dataset)
print("Num examples:", len(dataset))
print("Metadata fields:", dataset.metadata_fields)  # includes hospital / center id
print("\nSaved to:", CAMELYON17_DIR.resolve())

## 3. PANDA (Prostate cANcer graDe Assessment)

Hosted as a **Kaggle competition dataset** — requires a free Kaggle account, an API token, and
one-time acceptance of the competition rules (Kaggle blocks the download otherwise).

### One-time setup (per machine / server user)
1. Create a Kaggle account if you don't have one: https://www.kaggle.com
2. Go to **Account → Create New API Token** → this downloads `kaggle.json`
3. Upload `kaggle.json` to the server and run the cell below to place it correctly
4. Visit https://www.kaggle.com/c/prostate-cancer-grade-assessment/rules and click **"I Understand and Accept"**
   (Kaggle will reject the download via API until this is done, even with a valid token)

### Size warning
The full PANDA release (raw whole-slide `.tiff` files + mask files) is **several hundred GB**.
For a 3-day course, strongly consider downloading a curated subset (e.g. a stratified sample of
150–300 slides across ISUP grades) rather than the full ~11,000-slide release — see the subsetting
cell at the end of this section.

In [ ]:
# Place your kaggle.json (from Kaggle account settings) at ~/.kaggle/kaggle.json
# Uncomment and run once if you haven't already placed it manually:

# import shutil
# shutil.copy("/path/to/your/kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
# os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

!kaggle competitions list -s "prostate cancer grade"  # sanity check that auth works

In [ ]:
# Full download (several hundred GB — make sure DATA_ROOT has room, and that you've
# accepted the competition rules on the Kaggle website first).

!kaggle competitions download -c prostate-cancer-grade-assessment -p {str(PANDA_DIR)}

# Unzip (competition ships as a single large zip)
import zipfile

zip_path = PANDA_DIR / "prostate-cancer-grade-assessment.zip"
if zip_path.exists():
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(PANDA_DIR)
    print("Extracted to:", PANDA_DIR.resolve())
else:
    print("Zip not found — check the download step above for errors.")

### Lighter alternative: curated subset only

If you don't want to pull the full release, download just `train.csv` (labels + metadata, tiny)
first, pick a stratified sample of slide IDs across ISUP grades, then fetch only those slides'
`.tiff` files individually via the Kaggle API's file-level download. This keeps disk usage to a
few GB instead of hundreds.

In [ ]:
# 1. Grab just the label file first (tiny)
!kaggle competitions download -c prostate-cancer-grade-assessment -f train.csv -p {str(PANDA_DIR)}
train_df = pd.read_csv(PANDA_DIR / "train.csv")
print(train_df["isup_grade"].value_counts().sort_index())

# 2. Stratified sample — e.g. ~30 slides per ISUP grade (0-5) = ~180 slides total
N_PER_GRADE = 30
subset = train_df.groupby("isup_grade", group_keys=False).apply(
    lambda x: x.sample(min(N_PER_GRADE, len(x)), random_state=42)
)
subset.to_csv(PANDA_DIR / "train_subset.csv", index=False)
print(f"\nSelected {len(subset)} slides across {subset['isup_grade'].nunique()} grades.")

# 3. Download only those slides' WSI files (train_images/{image_id}.tiff)
# Kaggle's API doesn't support per-file glob downloads for competition datasets directly,
# so the practical approach is to download the full train_images.zip once, then delete
# unneeded slides — OR use the Kaggle API's dataset-level file listing if this competition
# exposes one. Check current options with:
!kaggle competitions files -c prostate-cancer-grade-assessment

## 4. Verify what you've got

In [ ]:
import subprocess

for name, d in [
    ("PathMNIST", PATHMNIST_DIR),
    ("CAMELYON17-WILDS", CAMELYON17_DIR),
    ("PANDA", PANDA_DIR),
]:
    if d.exists():
        result = subprocess.run(["du", "-sh", str(d)], capture_output=True, text=True)
        size = result.stdout.split()[0] if result.stdout else "?"
        print(f"{name:20s} {size:>8s}   {d}")
    else:
        print(f"{name:20s} {'—':>8s}   (not downloaded)")

## Next steps

- **PathMNIST**: ready to use as-is — images are already patch-sized and labeled.
- **CAMELYON17-WILDS**: ready to use — patches + hospital/domain metadata are already aligned.
- **PANDA**: raw WSIs need tiling/patch extraction before they can go through UNI2 — plan to
  run a tiling step (e.g. via [OpenSlide](https://openslide.org/) or
  [CLAM](https://github.com/mahmoodlab/CLAM)'s preprocessing scripts) centrally on the server
  *before* the course, not live during Day 1.
